In [16]:
%pip install prettytable

Note: you may need to restart the kernel to use updated packages.


In [17]:

import sqlite3
from prettytable import PrettyTable

DB_NAME = "expense_sharing.db"

In [18]:
# 1. DATABASE SETUP
def setup_database():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            user_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT UNIQUE )""")
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS expenses (
            expense_id INTEGER PRIMARY KEY AUTOINCREMENT,
            description TEXT,
            total_amount INTEGER,
            payer_id INTEGER,
            expense_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP ) """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS expense_shares (
            share_id INTEGER PRIMARY KEY AUTOINCREMENT,
            expense_id INTEGER,
            user_id INTEGER,
            share_amount INTEGER ) """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS settlements (
            settlement_id INTEGER PRIMARY KEY AUTOINCREMENT,
            payer_id INTEGER,
            receiver_id INTEGER,
            amount INTEGER,
            settlement_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP )""")
    conn.commit()
    conn.close()
    print("Database created successfully!")

In [19]:
# 2. ADD FRIEND
def add_friend():
    name = input("Enter friend name: ").strip()
    if not name:
        print("Name cannot be empty.")
        return
    conn = sqlite3.connect(DB_NAME)
    try:
        conn.execute(
            "INSERT INTO users (name) VALUES (?)",
            (name,))
        conn.commit()
        print("Friend added successfully!")
    except sqlite3.IntegrityError:
        print("Friend already exists.")
    conn.close()

In [20]:
# 3. VIEW FRIENDS
def view_friends():
    conn = sqlite3.connect(DB_NAME)
    users = conn.execute(
        "SELECT user_id, name FROM users"
    ).fetchall()
    conn.close()
    table = PrettyTable(["ID", "Friend Name"])
    for user in users:
        table.add_row(user)
    print(table if users else "No friends found.")

In [21]:
# 4. ADD EXPENSE
def add_expense():
    conn = sqlite3.connect(DB_NAME)
    users = conn.execute(
        "SELECT user_id, name FROM users"
    ).fetchall()
    if not users:
        print("Add friends first!")
        conn.close()
        return
    print("\nFriends:")
    for i, user in enumerate(users, 1):
        print(i, user[1])
    try:
        description = input("Enter expense description: ")
        total = int(round(float(input("Enter amount in rupees: ")) * 100))
        if total <= 0:
            print("Amount must be greater than zero.")
            conn.close()
            return
        payer_no = int(input("Who paid? Enter friend number: "))
        if payer_no < 1 or payer_no > len(users):
            print("Invalid friend number.")
            conn.close()
            return
        payer_id = users[payer_no - 1][0]
        print("Enter participant numbers separated by commas.")
        print("Example: 1,2,3")
        numbers = input("Participants: ").split(",")
        selected = []
        for number in numbers:
            number = int(number.strip())
            if number < 1 or number > len(users):
                raise ValueError
            selected.append(users[number - 1])
        if not selected:
            print("Select at least one participant.")
            conn.close()
            return
        selected_ids = [user[0] for user in selected]
        if len(set(selected_ids)) != len(selected_ids):
            print("Duplicate participants are not allowed.")
            conn.close()
            return
        print("\n1. Equal split")
        print("2. Custom split")
        choice = input("Choose split type: ")
        shares = []
        if choice == "1":
            share, remainder = divmod(total, len(selected))
            for i in range(len(selected)):
                amount = share + (1 if i < remainder else 0)
                shares.append(amount)
        elif choice == "2":
            for user in selected:
                amount = int(round(
                    float(input(
                        "Enter share for " + user[1] + ": "
                    )) * 100
                ))
                if amount < 0:
                    print("Share cannot be negative.")
                    conn.close()
                    return
                shares.append(amount)
            if sum(shares) != total:
                print("Total shares must equal the expense amount.")
                conn.close()
                return
        else:
            print("Invalid split choice.")
            conn.close()
            return
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO expenses
            (description, total_amount, payer_id)
            VALUES (?, ?, ?)
        """, (description, total, payer_id))
        expense_id = cursor.lastrowid
        for i, user in enumerate(selected):
            cursor.execute("""
                INSERT INTO expense_shares
                (expense_id, user_id, share_amount)
                VALUES (?, ?, ?)
            """, (expense_id, user[0], shares[i]))
        conn.commit()
        print("Expense added successfully!")
    except ValueError:
        print("Invalid input. Please try again.")
    finally:
        conn.close()

In [22]:
# 5. VIEW ALL EXPENSES
def view_expenses():
    conn = sqlite3.connect(DB_NAME)
    rows = conn.execute("""
        SELECT e.expense_id, e.description,
               e.total_amount, u.name, e.expense_date
        FROM expenses e
        JOIN users u ON e.payer_id = u.user_id
        ORDER BY e.expense_id DESC
    """).fetchall()
    conn.close()
    table = PrettyTable(
        ["ID", "Description", "Amount", "Paid By", "Date"]
    )
    for row in rows:
        table.add_row([
            row[0], row[1],
            "Rs. {:.2f}".format(row[2] / 100),
            row[3], row[4]
        ])
    print(table if rows else "No expenses found.")

In [23]:
# 6. INDIVIDUAL TRANSACTION HISTORY
def view_history():
    conn = sqlite3.connect(DB_NAME)
    users = conn.execute(
        "SELECT user_id, name FROM users"
    ).fetchall()
    if not users:
        print("No friends found.")
        conn.close()
        return
    for i, user in enumerate(users, 1):
        print(i, user[1])
    try:
        number = int(input("Select friend number: "))
        if number < 1 or number > len(users):
            print("Invalid selection.")
            conn.close()
            return
        user_id, name = users[number - 1]
        rows = conn.execute("""
            SELECT e.description, e.total_amount,
                   payer.name, s.share_amount, e.expense_date
            FROM expense_shares s
            JOIN expenses e ON s.expense_id = e.expense_id
            JOIN users payer ON e.payer_id = payer.user_id
            WHERE s.user_id = ?
            ORDER BY e.expense_id DESC
        """, (user_id,)).fetchall()
        table = PrettyTable(
            ["Expense", "Total", "Paid By", "Your Share", "Date"]
        )
        for row in rows:
            table.add_row([
                row[0],
                "Rs. {:.2f}".format(row[1] / 100),
                row[2],
                "Rs. {:.2f}".format(row[3] / 100),
                row[4]
            ])
        print("\nHistory of", name)
        print(table if rows else "No expense history.")
        payments = conn.execute("""
            SELECT receiver.name, s.amount, s.settlement_date
            FROM settlements s
            JOIN users receiver ON s.receiver_id = receiver.user_id
            WHERE s.payer_id = ?
            UNION ALL
            SELECT payer.name, -s.amount, s.settlement_date
            FROM settlements s
            JOIN users payer ON s.payer_id = payer.user_id
            WHERE s.receiver_id = ?
        """, (user_id, user_id)).fetchall()

        payment_table = PrettyTable(
            ["Other Person", "Amount", "Date"]
        )
        for person, amount, date in payments:
            direction = "Paid" if amount > 0 else "Received"
            payment_table.add_row([
                person,
                direction + " Rs. {:.2f}".format(abs(amount) / 100),
                date
            ])
        print("\nSettlement History")
        print(payment_table if payments else "No settlements found.")
    except ValueError:
        print("Enter a valid number.")
    finally:
        conn.close()


In [24]:
# 7. CALCULATE BALANCES
def calculate_balances():
    conn = sqlite3.connect(DB_NAME)
    users = conn.execute(
        "SELECT user_id, name FROM users"
    ).fetchall()
    balances = {}
    for user_id, name in users:
        balances[user_id] = [name, 0]
    expenses = conn.execute(
        "SELECT payer_id, total_amount FROM expenses"
    ).fetchall()
    for payer_id, amount in expenses:
        balances[payer_id][1] += amount
    shares = conn.execute(
        "SELECT user_id, share_amount FROM expense_shares"
    ).fetchall()
    for user_id, amount in shares:
        balances[user_id][1] -= amount
    payments = conn.execute(
        "SELECT payer_id, receiver_id, amount FROM settlements"
    ).fetchall()
    for payer, receiver, amount in payments:
        balances[payer][1] += amount
        balances[receiver][1] -= amount
    conn.close()
    return balances

In [25]:

def show_balances(suggest=False):
    balances = calculate_balances()
    table = PrettyTable(["Friend", "Balance"])
    for name, amount in balances.values():
        if amount > 0:
            status = "Gets back"
        elif amount < 0:
            status = "Owes"
        else:
            status = "Settled"
        table.add_row([name, status + " Rs. {:.2f}".format(abs(amount) / 100)])
    print(table)
    if suggest:
        debtors = [[user_id, data[0], -data[1]] for user_id, data in balances.items() if data[1] < 0]
        creditors = [[user_id, data[0], data[1]] for user_id, data in balances.items() if data[1] > 0]
        i = 0
        j = 0
        while i < len(debtors) and j < len(creditors):
            amount = min(debtors[i][2], creditors[j][2])
            print(debtors[i][1], "pays", creditors[j][1], "Rs. {:.2f}".format(amount / 100))
            debtors[i][2] -= amount
            creditors[j][2] -= amount
            if debtors[i][2] == 0:
                i += 1
            if creditors[j][2] == 0:
                j += 1
        if not debtors and not creditors:
            print("All balances are settled.")

In [26]:

def record_payment():
    balances = calculate_balances()
    debtors = [(user_id, data[0], -data[1]) for user_id, data in balances.items() if data[1] < 0]
    creditors = [(user_id, data[0], data[1]) for user_id, data in balances.items() if data[1] > 0]
    if not debtors or not creditors:
        print("No pending payments.")
        return
    print("People who owe money:")
    for i, person in enumerate(debtors, 1):
        print(i, person[1])
    try:
        payer_no = int(input("Who is paying? Enter number: "))
        if payer_no < 1 or payer_no > len(debtors):
            print("Invalid selection.")
            return
        payer_id, payer_name, debt = debtors[payer_no - 1]
        print("People who should receive money:")
        for i, person in enumerate(creditors, 1):
            print(i, person[1])
        receiver_no = int(input("Who is receiving? Enter number: "))
        if receiver_no < 1 or receiver_no > len(creditors):
            print("Invalid selection.")
            return
        receiver_id, receiver_name, credit = creditors[receiver_no - 1]
        maximum = min(debt, credit)
        print("Maximum payment: Rs. {:.2f}".format(maximum / 100))
        amount = int(round(float(input("Enter payment amount: ")) * 100))
        if amount <= 0 or amount > maximum:
            print("Invalid payment amount.")
            return
        conn = sqlite3.connect(DB_NAME)
        conn.execute(
            "INSERT INTO settlements (payer_id, receiver_id, amount) VALUES (?, ?, ?)",
            (payer_id, receiver_id, amount)
        )
        conn.commit()
        conn.close()
        print(payer_name, "paid", receiver_name, "Rs. {:.2f}".format(amount / 100))
        print("Payment recorded successfully!")
    except ValueError:
        print("Invalid input.")

In [27]:

def main():
    while True:
        print("\n========== GOOGLE PAY EXPENSE SHARING ==========")
        print("1. Add friend")
        print("2. View friends")
        print("3. Add expense")
        print("4. View all expenses")
        print("5. View individual transaction history")
        print("6. Display balance summary")
        print("7. Suggest payments")
        print("8. Record a settlement payment")
        print("9. Exit")

        choice = input("Enter your choice: ")

        if choice == "1":
            add_friend()
        elif choice == "2":
            view_friends()
        elif choice == "3":
            add_expense()
        elif choice == "4":
            view_expenses()
        elif choice == "5":
            view_history()
        elif choice == "6":
            show_balances()
        elif choice == "7":
            show_balances(True)
        elif choice == "8":
            record_payment()
        elif choice == "9":
            print("Thank you for using Google Pay Expense Sharing!")
            break
        else:
            print("Invalid choice. Select 1 to 9.")


In [ ]:
setup_database()
main()

Database created successfully!

========== GOOGLE PAY EXPENSE SHARING ==========
1. Add friend
2. View friends
3. Add expense
4. View all expenses
5. View individual transaction history
6. Display balance summary
7. Suggest payments
8. Record a settlement payment
9. Exit


Enter your choice:  7


+--------+-----------------------+
| Friend |        Balance        |
+--------+-----------------------+
| alice  |  Gets back Rs. 166.66 |
|  bob   |    Owes Rs. 883.33    |
| carol  | Gets back Rs. 1800.00 |
| nethra |    Owes Rs. 333.33    |
| aperna |    Owes Rs. 750.00    |
+--------+-----------------------+
bob pays alice Rs. 166.66
bob pays carol Rs. 716.67
nethra pays carol Rs. 333.33
aperna pays carol Rs. 750.00

========== GOOGLE PAY EXPENSE SHARING ==========
1. Add friend
2. View friends
3. Add expense
4. View all expenses
5. View individual transaction history
6. Display balance summary
7. Suggest payments
8. Record a settlement payment
9. Exit


Enter your choice:  8


People who owe money:
1 bob
2 nethra
3 aperna


Who is paying? Enter number:  1


People who should receive money:
1 alice
2 carol


Who is receiving? Enter number:  1


Maximum payment: Rs. 166.66


Enter payment amount:  166.66


bob paid alice Rs. 166.66
Payment recorded successfully!

========== GOOGLE PAY EXPENSE SHARING ==========
1. Add friend
2. View friends
3. Add expense
4. View all expenses
5. View individual transaction history
6. Display balance summary
7. Suggest payments
8. Record a settlement payment
9. Exit
